# 006: LSTM Gate Equations — Forget, input, output gates and cell state from scratch

## FULL LSTM EQUATIONS
$$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f) \quad \text{(forget gate)}$$
$$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i) \quad \text{(input gate)}$$
$$\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c) \quad \text{(candidate cell)}$$
$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \quad \text{(cell state update)}$$
$$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o) \quad \text{(output gate)}$$
$$h_t = o_t \odot \tanh(c_t) \quad \text{(hidden state)}$$
---


In [ ]:
import numpy as np

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

class LSTMCell:
    """Single LSTM cell from scratch with all four gates."""
    def __init__(self, input_dim, hidden_dim):
        n = hidden_dim
        m = input_dim
        scale = 0.01
        # Concatenated weight matrices: [h_{t-1}, x_t] has dim (n + m)
        self.Wf = np.random.randn(n, n + m) * scale  # Forget gate
        self.Wi = np.random.randn(n, n + m) * scale  # Input gate
        self.Wc = np.random.randn(n, n + m) * scale  # Candidate
        self.Wo = np.random.randn(n, n + m) * scale  # Output gate
        self.bf = np.zeros((n, 1))
        self.bi = np.zeros((n, 1))
        self.bc = np.zeros((n, 1))
        self.bo = np.zeros((n, 1))
        self.hidden_dim = n

    def forward(self, x_t, h_prev, c_prev):
        """Single time step forward pass."""
        concat = np.vstack([h_prev, x_t])  # (n+m, 1)
        f_t = sigmoid(self.Wf @ concat + self.bf)         # Forget gate
        i_t = sigmoid(self.Wi @ concat + self.bi)          # Input gate
        c_tilde = np.tanh(self.Wc @ concat + self.bc)     # Candidate
        c_t = f_t * c_prev + i_t * c_tilde                # Cell state
        o_t = sigmoid(self.Wo @ concat + self.bo)          # Output gate
        h_t = o_t * np.tanh(c_t)                           # Hidden state
        return h_t, c_t, {"f": f_t, "i": i_t, "o": o_t}



In [ ]:
print("--- LSTM Cell Forward Pass ---")
np.random.seed(42)
lstm = LSTMCell(input_dim=3, hidden_dim=4)
h = np.zeros((4, 1))
c = np.zeros((4, 1))

sequence = [np.random.randn(3, 1) for _ in range(5)]
for t, x in enumerate(sequence):
    h, c, gates = lstm.forward(x, h, c)
    print(f"t={t} | h_norm={np.linalg.norm(h):.4f} | c_norm={np.linalg.norm(c):.4f} | "
          f"forget={gates['f'].mean():.3f} | input={gates['i'].mean():.3f} | output={gates['o'].mean():.3f}")
